# 08 · Gold facts
Headcount (periodic snapshot), attrition (accumulating snapshot) and learning (transaction) facts (module 07 §3).

✅ Expect **116,999 / 688 / 17,923** rows.

In [ ]:
from hcm_utils import load_config, rebuild

dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")
cfg = load_config()
s, g = f"{catalog}.silver", f"{catalog}.gold"
snap_start, snap_end = cfg["snapshot"]["start_date"], cfg["snapshot"]["end_date"]

In [ ]:
# headcount: periodic snapshot
n_hc = rebuild(f"{g}.factworkforceheadcount", f"""
SELECT d.DateKey            AS SnapshotDateKey,
       w.WorkerKey, o.OrganizationKey, j.JobKey, gr.GradeKey, l.LocationKey, le.LegalEntityKey,
       a.AssignmentId, a.AssignmentCategory, a.AssignmentStatus, a.IsPrimary,
       1                                                              AS HeadcountTotal,
       CASE WHEN a.IsPrimary THEN 1 ELSE 0 END                        AS PrimaryHeadcount,
       CASE WHEN a.IsPrimary AND w.PersonType = 'EMP' THEN 1 ELSE 0 END AS EmployeeHeadcount,
       CASE WHEN a.IsPrimary AND w.PersonType = 'CWK' THEN 1 ELSE 0 END AS ContingentHeadcount
FROM {s}.hcmassignment a
JOIN {g}.dimdate d
  ON d.IsMonthEnd
 AND d.Date BETWEEN DATE'{snap_start}' AND DATE'{snap_end}'
 AND d.Date BETWEEN a.EffectiveStartDate AND a.EffectiveEndDate
JOIN {g}.dimworker       w  ON w.PersonId       = a.PersonId
JOIN {g}.dimorganization o  ON o.OrganizationId = a.OrganizationId
JOIN {g}.dimjob          j  ON j.JobId          = a.JobId
JOIN {g}.dimgrade        gr ON gr.GradeId       = a.GradeId
JOIN {g}.dimlocation     l  ON l.LocationId     = a.LocationId
JOIN {g}.dimlegalentity  le ON le.LegalEntityId = a.LegalEntityId
""")

In [ ]:
# attrition: accumulating snapshot
n_attr = rebuild(f"{g}.factworkforceattrition", f"""
WITH term AS (
  SELECT * FROM {s}.hcmperiodsofservice WHERE IsTerminated
),
asg_at_exit AS (
  SELECT t.PeriodOfServiceId, a.OrganizationId, a.JobId, a.GradeId, a.LocationId, a.LegalEntityId,
         row_number() OVER (PARTITION BY t.PeriodOfServiceId ORDER BY a.EffectiveStartDate DESC) AS rn
  FROM term t
  JOIN {s}.hcmassignment a
    ON a.PersonId = t.PersonId AND a.IsPrimary AND a.EffectiveStartDate <= t.ActualTerminationDate
)
SELECT CAST(date_format(t.ActualTerminationDate, 'yyyyMMdd') AS INT) AS TerminationDateKey,
       w.WorkerKey, o.OrganizationKey, j.JobKey, gr.GradeKey, l.LocationKey, le.LegalEntityKey,
       t.PeriodOfServiceId,
       t.DateStart             AS HireDate,
       t.ActualTerminationDate AS TerminationDate,
       t.TenureMonths          AS TenureAtExitMonths,
       t.TerminationType,
       t.LeavingReason,
       1                                                         AS AttritionCount,
       CASE WHEN t.TerminationType = 'Voluntary' THEN 1 ELSE 0 END AS VoluntaryAttritionCount
FROM term t
JOIN asg_at_exit x ON x.PeriodOfServiceId = t.PeriodOfServiceId AND x.rn = 1
JOIN {g}.dimworker       w  ON w.PersonId       = t.PersonId
JOIN {g}.dimorganization o  ON o.OrganizationId = x.OrganizationId
JOIN {g}.dimjob          j  ON j.JobId          = x.JobId
JOIN {g}.dimgrade        gr ON gr.GradeId       = x.GradeId
JOIN {g}.dimlocation     l  ON l.LocationId     = x.LocationId
JOIN {g}.dimlegalentity  le ON le.LegalEntityId = x.LegalEntityId
""")

In [ ]:
# learning: transaction fact
n_lrn = rebuild(f"{g}.factlearningactivity", f"""
SELECT w.WorkerKey,
       CAST(date_format(la.EnrollmentDate, 'yyyyMMdd') AS INT) AS EnrollmentDateKey,
       CAST(date_format(la.CompletionDate, 'yyyyMMdd') AS INT) AS CompletionDateKey,
       la.EnrollmentId, la.CourseId, la.CourseName, la.CourseCategory,
       la.CompletionStatus, la.CertificateId,
       1                                         AS EnrollmentCount,
       CASE WHEN la.IsCompleted THEN 1 ELSE 0 END AS CompletionCount,
       CASE WHEN la.IsMandatory THEN 1 ELSE 0 END AS MandatoryTrainingCount,
       la.Hours, la.IsCompleted, la.IsMandatory
FROM {s}.hcmlearningactivity la
JOIN {g}.dimworker w ON w.PersonId = la.PersonId
""")

In [ ]:
# guardrails: inner joins must not drop rows
expected_hc = spark.sql(f"""
  SELECT COUNT(*) FROM {s}.hcmassignment a JOIN {g}.dimdate d
    ON d.IsMonthEnd AND d.Date BETWEEN DATE'{snap_start}' AND DATE'{snap_end}'
   AND d.Date BETWEEN a.EffectiveStartDate AND a.EffectiveEndDate""").first()[0]
assert n_hc == expected_hc, f"headcount lost rows in key lookups: {n_hc} vs {expected_hc}"
assert n_attr == spark.table(f"{s}.hcmperiodsofservice").filter("IsTerminated").count()
assert n_lrn == spark.table(f"{s}.hcmlearningactivity").count()
print(dict(headcount=n_hc, attrition=n_attr, learning=n_lrn))

In [ ]:
# Table and column comments: Genie reads these
for stmt in [
    f"""COMMENT ON TABLE {g}.factworkforceheadcount IS
        'Month-end workforce snapshot. Grain: one row per assignment per month-end. Use SUM(PrimaryHeadcount) for headcount; HeadcountTotal counts assignments.'""",
    f"""ALTER TABLE {g}.factworkforceheadcount ALTER COLUMN PrimaryHeadcount
        COMMENT '1 if primary assignment. SUM = distinct worker headcount.'""",
    f"""COMMENT ON TABLE {g}.factworkforceattrition IS
        'One row per termination. TerminationType: Voluntary/Involuntary. Attrition rate = leavers / average monthly headcount.'""",
    f"""COMMENT ON TABLE {g}.factlearningactivity IS
        'One row per course enrollment. Mandatory compliance % = completed mandatory / mandatory enrollments.'""",
]:
    spark.sql(stmt)